In [8]:
# guarda esto como extraer_utensilios.py
import requests
from bs4 import BeautifulSoup
import json

def parse_html(html):
    soup = BeautifulSoup(html, "html.parser")
    # apuntar al contenedor principal (prueba varias rutas)
    container = soup.select_one("#page #content #con-full #primary #entry-content") \
                or soup.select_one("#entry-content") \
                or soup.select_one(".entry-content") \
                or soup

    items = []
    # tomar todos los <li> dentro del contenedor
    for li in container.find_all("li"):
        # si el <li> está vacío, saltar
        if not li.text.strip():
            continue

        # preferir el atributo title del <a> si existe
        a = li.find("a", recursive=False)  # enlace directo hijo
        name = None
        url = None
        if a:
            url = a.get("href")
            title = a.get("title")
            if title and title.strip():
                name = title.strip()
            else:
                # si no hay title usar solo el texto del enlace
                name = a.get_text(strip=True)

        # Si no hay <a> hijo directo o no produjo nombre, extraer texto del <li>
        if not name:
            # evitamos incluir textos de <ul> anidados: clonamos y eliminamos sub-uls
            clone = BeautifulSoup(str(li), "html.parser")
            for sub in clone.find_all("ul"):
                sub.decompose()
            name = clone.get_text(strip=True)

        # limpiar espacios redundantes
        name = " ".join(name.split())
        if not name:
            continue

        items.append({"nombre": name, "url": url})

    # deduplicar preservando primer ocurrencia
    seen = set()
    dedup = []
    for it in items:
        key = it["nombre"].lower()
        if key not in seen:
            seen.add(key)
            dedup.append(it)

    return dedup

def from_url(url):
    # ignorar robots (petición directa)
    r = requests.get(url, headers={"User-Agent":"Mozilla/5.0"})
    r.raise_for_status()
    return parse_html(r.text)

def from_html_string(html_string):
    return parse_html(html_string)

if __name__ == "__main__":
    # Opción A: descargar desde la web
    # resultado = from_url("https://mundococina.es/utensilios-de-cocina-nombres-y-tipos/")

    # Opción B: usar el HTML que pegaste (pegarlo en la variable html_input)
    # html_input = """... aquí tu bloque <div class="entry-content"> ... """
    # resultado = from_html_string(html_input)

    # Por defecto intento descargar
    try:
        resultado = from_url("https://mundococina.es/utensilios-de-cocina-nombres-y-tipos/")
    except Exception as e:
        print("Error al descargar:", e)
        resultado = []

    # guardar JSON
    with open("utensilios.json", "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=2)

    print(f"Guardados {len(resultado)} utensilios en utensilios.json")


Guardados 350 utensilios en utensilios.json
